# Phase 5 Analysis: Bootstrap CI on turns-to-convergence

**Requirement:** ALAB-03 (bootstrap 95% CI on the headline quantitative claim).

**Kernel registration (run once in your venv):**

```
python -m ipykernel install --user --name=conversational-clustering
```

Then select the `conversational-clustering` kernel in Jupyter.

This notebook reuses `src/analysis.py:compute_bootstrap_ci` (the SAME function the CLI `examples/compute_ci.py` uses — D-15) and queries the `experiments` table via `src/db/experiments.py:query` (CLAUDE.md: all SQL stays in `src/db/`).

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt

from src.analysis import compute_bootstrap_ci
from src.db import experiments as exp_db
from src.db.connection import connect, init_schema

In [ ]:
# Query all experiments (optionally filter by dataset or oracle_type)
db = connect()
init_schema(db)
try:
    rows = exp_db.query(db)
finally:
    db.close()

print(f'Loaded {len(rows)} experiments')
for r in rows[:5]:
    print(f'  {r.strategy_id:<22} persona={r.persona_id:<10} seed={r.seed} oracle={r.oracle_type} turns={r.total_turns}')

In [ ]:
# Group by strategy_id and compute 95% bootstrap CI on turns-to-convergence
groups = {}
for r in rows:
    if r.total_turns is None:
        continue  # skip unsealed rows
    groups.setdefault(r.strategy_id, []).append(float(r.total_turns))

for strategy_id, values in sorted(groups.items()):
    if not values:
        continue
    mean_val = float(np.mean(values))
    lo, hi = compute_bootstrap_ci(values, n_bootstrap=10000, ci=0.95, seed=0)
    print(f'{strategy_id:<22} mean={mean_val:6.2f}  95% CI=[{lo:.2f}, {hi:.2f}]  N={len(values)}')

In [ ]:
# Plot bootstrap distributions per strategy (visual inspection)
fig, ax = plt.subplots(figsize=(10, 5))
for strategy_id, values in sorted(groups.items()):
    if len(values) < 2:
        continue
    arr = np.asarray(values, dtype=np.float64)
    rng = np.random.default_rng(0)
    idx = rng.integers(low=0, high=len(arr), size=(10000, len(arr)))
    means = arr[idx].mean(axis=1)
    ax.hist(means, bins=40, alpha=0.4, label=strategy_id)
ax.set_xlabel('Bootstrap-resampled mean turns-to-convergence')
ax.set_ylabel('Density')
ax.set_title('Bootstrap distributions per strategy (ALAB-03)')
ax.legend()
plt.tight_layout()
plt.show()